# M3 채널 확대 실험 — 64 → 128 → 128

**런타임 → 런타임 유형 변경 → GPU**를 선택하고, 새 런타임에서 위에서부터 실행하세요.

- 증류 없이 정답(CE)으로만 학습합니다. 교사 가중치는 받지 않습니다.
- Google 포함 20서비스 × 서비스당 200/1,000개 × seed 201/202 = **새 학습 4회**.
- 채널 이외 설정은 중간발표 M3와 동일: 학습률 0.003, 50epoch, batch128, AdamW, warm-up5 + cosine.
- 1월 학습, **2월 검증**으로 선택합니다. 이 노트북은 4·5월 평가 점수를 계산하지 않습니다.
- 공식 데이터 약 **3.54GB**를 다운로드합니다. 준비 파일 포함 디스크는 **30GB 이상을 계획값**으로 잡으세요.
- 데이터 준비는 CPU 작업입니다. 준비 중 RAM이 부족하면 더 큰 RAM 런타임이 필요합니다.
- 결과는 `/content`에 저장됩니다. **런타임을 삭제하기 전에 마지막 셀로 ZIP을 내려받으세요.**

기준 저장소 commit: `a78399589391a4807df8ede9ef8205eac9d2e466`.
새 실험 코드는 이 노트북 안에 포함돼 있어 GitHub에 올리지 않아도 실행할 수 있습니다.


In [ ]:
import sys, subprocess, os, json, hashlib, shutil
from pathlib import Path

ROOT = Path('/content/helloworld-m3-width')
COMMIT = 'a78399589391a4807df8ede9ef8205eac9d2e466'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/RealOctavision/helloworld.git', str(ROOT)], check=True)
    if subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() != COMMIT:
        subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', COMMIT], check=True)
        subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() == COMMIT, '다른 저장소 버전입니다. 별도 새 폴더를 사용하세요.'

# Install before importing NumPy/PyTorch in a fresh runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy==2.1.3', 'cesnet-models==0.4.1', 'tables==3.11.1'], check=True)
os.chdir(ROOT)
import torch
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('기존 실험 환경: A100 / Torch 2.11.0+cu130. 다른 환경은 결과에 기록됩니다.')
print('남은 디스크 GB:', round(shutil.disk_usage(ROOT).free / 1e9, 1))


## 1. 확대 모델 코드와 작은 동작 테스트

다음 셀은 이번에 작성한 코드를 새 실험 폴더에 저장합니다. 중간발표 학습·모델 파일은 수정하지 않습니다.
합성 데이터 CPU 테스트 점수는 연구 결과가 아닙니다.


In [ ]:
EXP = ROOT / "experiments/2026-11-05_m3_width"
EXP.mkdir(parents=True, exist_ok=True)
SOURCES = {'run.py': '"""CE-only 64/128/128 student, paired with the published M3 samples.\n\nThe midterm files are read-only dependencies. Only February is scored here.\n"""\nimport os\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\', \':4096:8\')\n\nimport argparse\nimport fcntl\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport sys\nimport time\n\nHERE = Path(__file__).resolve().parent\nBASE = HERE.parent / \'2026-10-30_midterm_restart\'\nsys.path.insert(0, str(BASE))\n\nimport numpy as np\nimport torch\nfrom torch import nn\nimport data\nimport models\nimport training\n\nCOHORT = \'google_included20\'\nKIND = \'M3_wide\'\nCHANNELS = [64, 128, 128]\nBASE_HASHES = {\n    \'config.json\': \'f1b4de11292ae7bbd0d4a76f47deedf6f63798e8edc509e55a2a82f9ed8409fa\',\n    \'main_frozen.json\': \'e8f529a9d1e080c3a5c1faba10b20b439ab6221206173258aa51e74dd4197054\',\n    \'data.py\': \'9bfcaced8696fe9e4dd348e493a6a7931038753517e5620beb203731ac4734a4\',\n    \'models.py\': \'8e01ad21e877e7c9e54b69eb73f2283ebe1367a7a6392853d0583216dfbd16b1\',\n    \'training.py\': \'e25568e097d4b1d04465123554e3f54c18e971aa6030b51b04bdf336438fc955\',\n}\n\n\nclass WideStudent(models.Student):\n    def __init__(self, classes):\n        # Inherit the exact scaling, masking, ReLU and pooling implementation.\n        nn.Module.__init__(self)\n        self.drop_channel = None\n        self.blocks = nn.ModuleList([\n            nn.Conv1d(3, 64, 3, padding=1),\n            nn.Conv1d(64, 128, 3, padding=1),\n            nn.Conv1d(128, 128, 3, padding=1),\n        ])\n        self.head = nn.Linear(128, classes)\n\n\ndef make_wide_model(kind, classes, seed, cfg):\n    if kind != KIND or classes != 20:\n        raise ValueError(\'This experiment supports only M3_wide / 20 classes\')\n    torch.manual_seed(seed + 10000)\n    model = WideStudent(classes)\n    torch.manual_seed(seed + 20000)\n    model.head.reset_parameters()\n    if sum(p.numel() for p in model.parameters()) != 77204:\n        raise ValueError(\'Unexpected wide architecture\')\n    return model\n\n\ndef settings():\n    for name, expected in BASE_HASHES.items():\n        if data.digest(BASE / name) != expected:\n            raise ValueError(f\'Midterm dependency changed: {name}\')\n    cfg = data.load_config()\n    frozen = json.loads((BASE / \'main_frozen.json\').read_text())\n    cfg[\'training\'][\'epochs_frozen\'] = True\n    cfg[\'student\'][\'channels\'] = CHANNELS\n    cfg[\'student\'][\'params_20classes\'] = 77204\n    return cfg, frozen\n\n\ndef fit_wide(*args, **kwargs):\n    # Local, scoped factory substitution keeps the frozen training file intact.\n    original = training.make_model\n    training.make_model = make_wide_model\n    try:\n        return training.fit_network(KIND, *args, **kwargs)\n    finally:\n        training.make_model = original\n\n\ndef split_hashes(train, val):\n    return {\n        f\'{prefix}_{field}_sha256\': training.hash_bytes(split[key])\n        for prefix, split in [(\'train\', train), (\'validation\', val)]\n        for field, key in [(\'ids\', \'id\'), (\'input\', \'X\'), (\'labels\', \'y\')]\n    }\n\n\ndef baseline(seed, budget, hashes):\n    if budget == 200:\n        pattern = f\'main_recovered/training_*/extracted/main_training/fits/M3_s{seed}_n200/metrics.json\'\n    else:\n        pattern = f\'continuation_recovered/mixed_n1000/training_*/extracted/continuation/mixed_n1000/training/fits/M3_s{seed}_n1000/metrics.json\'\n    paths = list(BASE.glob(pattern))\n    if not paths:\n        return None\n    if len(paths) != 1:\n        raise ValueError(\'Multiple baseline records; resolve before comparing\')\n    old = json.loads(paths[0].read_text())\n    expected = dict(model=\'M3\', cohort=COHORT, seed=seed, budget=budget,\n                    lr=0.003, total_epochs=50, alpha=0.0, parameters=20180,\n                    config_sha256=BASE_HASHES[\'config.json\'])\n    expected.update({k: v for k, v in hashes.items() if k in [\n        \'train_ids_sha256\', \'train_input_sha256\', \'train_labels_sha256\',\n        \'validation_ids_sha256\']})\n    if any(old.get(k) != v for k, v in expected.items()):\n        raise ValueError(f\'Baseline/data mismatch: {paths[0]}\')\n    return dict(path=str(paths[0].relative_to(BASE)), validation=old[\'validation\'],\n                torch_version=old[\'torch_version\'], cuda_version=old[\'cuda_version\'],\n                device_name=old[\'device_name\'])\n\n\ndef bind_context(folder, context):\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / \'run_context.json\'\n    if path.exists():\n        if json.loads(path.read_text()) != context:\n            raise ValueError(f\'Run settings/data/environment changed: {folder}; use a new output directory\')\n    else:\n        if any(folder.iterdir()):\n            raise ValueError(f\'Unidentified existing files: {folder}\')\n        data.write_json(path, context)\n\n\ndef completed_result(folder):\n    path = folder / \'metrics.json\'\n    if not path.exists():\n        return None\n    result = json.loads(path.read_text())\n    for which in [\'best\', \'last\']:\n        if data.digest(folder / f\'{which}.pt\') != result[f\'{which}_checkpoint_sha256\']:\n            raise ValueError(f\'Corrupt {which} checkpoint: {folder}\')\n    return result\n\n\ndef train_score(model, train, device):\n    logits = training.logits_for(model, train[\'X\'], torch.device(device))\n    return training.metrics(train[\'y\'], logits.softmax(1).numpy(), 20)\n\n\ndef summarize(rows):\n    groups = {}\n    for budget in sorted({r[\'budget\'] for r in rows}):\n        selected = [r for r in rows if r[\'budget\'] == budget]\n        values = [r[\'validation\'][\'macro_f1\'] for r in selected]\n        deltas = [r[\'delta_validation_macro_f1_pp\'] for r in selected\n                  if r[\'delta_validation_macro_f1_pp\'] is not None]\n        groups[str(budget)] = dict(\n            seeds=[r[\'seed\'] for r in selected],\n            validation_macro_f1_mean=float(np.mean(values)),\n            validation_macro_f1_sample_std=float(np.std(values, ddof=1)) if len(values) > 1 else None,\n            paired_seed_count=len(deltas),\n            paired_delta_macro_f1_pp_mean=float(np.mean(deltas)) if deltas else None)\n    return groups\n\n\ndef run(args):\n    cfg, frozen = settings()\n    if args.dry_run:\n        print(json.dumps(dict(model=KIND, channels=CHANNELS, parameters=77204,\n            budgets=args.budgets, seeds=args.seeds, fits=len(args.budgets)*len(args.seeds),\n            lr=frozen[\'selected\'][\'M3\'], epochs=frozen[\'epochs\'],\n            training=cfg[\'training\'], evaluation=\'February validation only\'), indent=2))\n        return\n    if args.device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA unavailable; choose a GPU runtime (or explicitly --device cpu)\')\n    output = args.output.resolve()\n    if output == BASE or BASE in output.parents:\n        raise ValueError(\'Output must be outside the preserved midterm directory\')\n    output.mkdir(parents=True, exist_ok=True)\n    lock = (output / \'runner.lock\').open(\'a\')\n    try:\n        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        _run_locked(args, cfg, frozen, output)\n    finally:\n        lock.close()\n\n\ndef _run_locked(args, cfg, frozen, output):\n    deadline = time.monotonic() + args.minutes * 60\n    val = data.read_split(COHORT, \'validation\')\n    environment = dict(python=platform.python_version(), torch=str(torch.__version__),\n                       numpy=np.__version__, cuda=torch.version.cuda, device=args.device,\n                       device_name=torch.cuda.get_device_name(0) if args.device == \'cuda\' else platform.machine())\n    source_hashes = dict(BASE_HASHES, runner=data.digest(Path(__file__)))\n    rows = []\n    current = None\n\n    def status(state, error=None):\n        data.write_json(output / \'summary.json\', dict(state=state, current=current,\n            planned_fits=len(args.budgets)*len(args.seeds), completed_fits=len(rows),\n            environment=environment, runs=rows, by_budget=summarize(rows), error=error,\n            scores_are_fractions=True, delta_unit=\'percentage points\',\n            April_May_evaluated=False))\n\n    try:\n        status(\'running\')\n        for budget in args.budgets:\n            for seed in args.seeds:\n                current = f\'{KIND}_s{seed}_n{budget}\'\n                folder = output / current\n                train = data.read_split(COHORT, f\'main_seed{seed}\', budget)\n                hashes = split_hashes(train, val)\n                old = baseline(seed, budget, hashes)\n                context = dict(model=KIND, channels=CHANNELS, parameters=77204,\n                    cohort=COHORT, seed=seed, budget=budget, lr=frozen[\'selected\'][\'M3\'],\n                    epochs=frozen[\'epochs\'], effective_config=cfg, source_hashes=source_hashes,\n                    environment=environment, baseline=old, **hashes)\n                bind_context(folder, context)\n                result = completed_result(folder)\n                if result is None:\n                    if time.monotonic() >= deadline:\n                        raise TimeoutError(\'Session budget reached; rerun the same command to resume\')\n                    model, result = fit_wide(train, val, cfg, folder, COHORT, seed, budget,\n                        frozen[\'selected\'][\'M3\'], frozen[\'epochs\'], device=args.device,\n                        phase=\'main\', alpha=0.0, teacher_checkpoint=None,\n                        time_limit=max(0.001, deadline-time.monotonic()),\n                        resume_from=folder/\'last.pt\' if (folder/\'last.pt\').exists() else None)\n                else:\n                    model = make_wide_model(KIND, 20, seed, cfg).to(args.device)\n                    training.load_checkpoint(folder / \'best.pt\', model)\n                # The shared loop records per-epoch train loss. Score the selected\n                # checkpoint once on training data to diagnose the generalization gap.\n                diagnostic = folder / \'diagnostics.json\'\n                if diagnostic.exists():\n                    detail = json.loads(diagnostic.read_text())\n                    if detail[\'best_checkpoint_sha256\'] != result[\'best_checkpoint_sha256\']:\n                        raise ValueError(\'Diagnostics belong to a different checkpoint\')\n                else:\n                    detail = dict(best_checkpoint_sha256=result[\'best_checkpoint_sha256\'],\n                                  train_at_selected_checkpoint=train_score(model, train, args.device))\n                    data.write_json(diagnostic, detail)\n                row = dict(model=KIND, channels=CHANNELS, parameters=77204, seed=seed, budget=budget,\n                    selected_epoch=result[\'selected_epoch\'], validation=result[\'validation\'],\n                    train=detail[\'train_at_selected_checkpoint\'],\n                    updates=result[\'history\'][-1][\'updates\'],\n                    baseline_validation=None if old is None else old[\'validation\'],\n                    delta_validation_macro_f1_pp=None if old is None else\n                        100*(result[\'validation\'][\'macro_f1\']-old[\'validation\'][\'macro_f1\']),\n                    baseline_environment_matches=None if old is None else\n                        (old[\'torch_version\'] == environment[\'torch\'] and\n                         old[\'cuda_version\'] == environment[\'cuda\'] and\n                         old[\'device_name\'] == environment[\'device_name\']))\n                rows.append(row)\n                print(json.dumps({k: row[k] for k in [\'seed\', \'budget\', \'delta_validation_macro_f1_pp\',\n                    \'baseline_environment_matches\']}), flush=True)\n                del model, train\n                if args.device == \'cuda\':\n                    torch.cuda.empty_cache()\n                status(\'running\')\n        current = None\n        status(\'completed\')\n    except BaseException as exc:\n        status(\'budget_stopped\' if isinstance(exc, TimeoutError) else \'failed\', str(exc))\n        raise\n\n\ndef parse_args(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--budgets\', type=int, nargs=\'+\', choices=[200, 1000], default=[200, 1000])\n    p.add_argument(\'--seeds\', type=int, nargs=\'+\', choices=range(201, 206), default=[201, 202])\n    p.add_argument(\'--device\', choices=[\'cuda\', \'cpu\'], default=\'cuda\')\n    p.add_argument(\'--minutes\', type=float, default=45)\n    p.add_argument(\'--output\', type=Path, default=HERE/\'runs\')\n    p.add_argument(\'--dry-run\', action=\'store_true\')\n    args = p.parse_args(argv)\n    if not math.isfinite(args.minutes) or args.minutes <= 0:\n        p.error(\'--minutes must be finite and positive\')\n    if len(set(args.budgets)) != len(args.budgets) or len(set(args.seeds)) != len(args.seeds):\n        p.error(\'Duplicate budgets or seeds\')\n    return args\n\n\nif __name__ == \'__main__\':\n    run(parse_args())\n', 'smoke_test.py': '"""Synthetic CPU checks. No real traffic, teacher weights or test months."""\nimport copy\nimport json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom unittest import mock\n\nimport numpy as np\nimport torch\n\nimport run\n\n\ndef sample(budget, start):\n    rng = np.random.default_rng(start)\n    n = 20 * budget\n    x = np.zeros((n, 3, 30), dtype=np.int32)\n    x[:, 0, :12] = rng.integers(0, 500, (n, 12))\n    x[:, 1, :12] = rng.choice([-1, 1], (n, 12))\n    x[:, 2, :12] = rng.integers(1, 1501, (n, 12))\n    return dict(X=x, y=np.repeat(np.arange(20), budget), id=np.arange(start, start+n))\n\n\nclass WideTests(unittest.TestCase):\n    def test_architecture_mask_and_gradients(self):\n        cfg, _ = run.settings()\n        model = run.make_wide_model(run.KIND, 20, 201, cfg)\n        self.assertEqual(sum(p.numel() for p in model.parameters()), 77204)\n        self.assertEqual(sum(p.numel() for p in run.models.Student(20).parameters()), 20180)\n        x = torch.as_tensor(sample(1, 0)[\'X\'], dtype=torch.float32)\n        model.eval()\n        expected = model(x)\n        self.assertEqual(tuple(expected.shape), (20, 20))\n        # Padding must not introduce bias-derived features into pooled outputs.\n        self.assertTrue(torch.allclose(expected, model(torch.nn.functional.pad(x, (0, 10))), atol=1e-7))\n        self.assertTrue(torch.isfinite(model(torch.zeros_like(x))).all())\n        torch.nn.functional.cross_entropy(expected, torch.arange(20)).backward()\n        for p in model.parameters():\n            self.assertIsNotNone(p.grad)\n            self.assertTrue(torch.isfinite(p.grad).all())\n\n    def test_actual_training_and_exact_resume(self):\n        cfg, _ = run.settings()\n        cfg = copy.deepcopy(cfg)\n        cfg[\'training\'][\'batch_size\'] = 32\n        train, val = sample(3, 0), sample(2, 1000)\n        original_factory = run.training.make_model\n        with tempfile.TemporaryDirectory() as temp:\n            direct, resumed = Path(temp)/\'direct\', Path(temp)/\'resumed\'\n            kw = dict(cohort=run.COHORT, seed=201, budget=3, lr=0.003,\n                      epochs=3, device=\'cpu\', phase=\'smoke\', warmup_epochs=1)\n            initial = run.models.state_digest(run.make_wide_model(run.KIND, 20, 201, cfg))\n            model, result = run.fit_wide(train, val, cfg, direct, **kw)\n            self.assertNotEqual(run.models.state_digest(model), initial)\n            self.assertEqual(result[\'completed_epochs\'], 3)\n            self.assertEqual(result[\'teacher_prediction_seconds\'], 0)\n            self.assertFalse(result[\'performance_result\'])\n            self.assertIs(run.training.make_model, original_factory)\n            with self.assertRaises(TimeoutError):\n                run.fit_wide(train, val, cfg, resumed, time_limit=-1, **kw)\n            self.assertIs(run.training.make_model, original_factory)\n            restored, other = run.fit_wide(train, val, cfg, resumed,\n                                           resume_from=resumed/\'last.pt\', **kw)\n            self.assertEqual(run.models.state_digest(model), run.models.state_digest(restored))\n            self.assertEqual(result[\'validation\'], other[\'validation\'])\n            for which in [\'best\', \'last\']:\n                a = torch.load(direct/f\'{which}.pt\', weights_only=True)[\'model\']\n                b = torch.load(resumed/f\'{which}.pt\', weights_only=True)[\'model\']\n                self.assertTrue(all(torch.equal(a[k], b[k]) for k in a))\n            self.assertEqual(run.train_score(model, train, \'cpu\')[\'n\'], 60)\n\n    def test_context_and_corruption_guards(self):\n        with tempfile.TemporaryDirectory() as temp:\n            folder = Path(temp)/\'fit\'\n            run.bind_context(folder, {\'channels\': run.CHANNELS, \'seed\': 201})\n            run.bind_context(folder, {\'channels\': run.CHANNELS, \'seed\': 201})\n            with self.assertRaises(ValueError):\n                run.bind_context(folder, {\'channels\': run.CHANNELS, \'seed\': 202})\n            for which in [\'best\', \'last\']:\n                (folder/f\'{which}.pt\').write_bytes(b\'checkpoint\')\n            record = {f\'{which}_checkpoint_sha256\': run.data.digest(folder/f\'{which}.pt\')\n                      for which in [\'best\', \'last\']}\n            (folder/\'metrics.json\').write_text(json.dumps(record))\n            self.assertIsNotNone(run.completed_result(folder))\n            (folder/\'best.pt\').write_bytes(b\'corrupt\')\n            with self.assertRaises(ValueError):\n                run.completed_result(folder)\n\n    def test_four_run_plan_and_nested_data(self):\n        args = run.parse_args([])\n        self.assertEqual([(b, s) for b in args.budgets for s in args.seeds],\n                         [(200, 201), (200, 202), (1000, 201), (1000, 202)])\n        y = np.repeat(np.arange(20), 1100)\n        indices, rank = run.data.sample_per_class(y, 1000, 201, require_full=True)\n        self.assertEqual(len(indices[rank < 200]), 4000)\n        self.assertEqual(len(indices), 20000)\n        self.assertTrue(set(indices[rank < 200]) <= set(indices))\n        self.assertEqual(run.summarize([]), {})\n\n    def test_runner_four_fits_and_completed_reuse(self):\n        cfg, frozen = run.settings()\n        frozen = dict(frozen, epochs=1)\n        val = sample(1, 100000)\n\n        def read_split(cohort, name, budget=None):\n            self.assertEqual(cohort, run.COHORT)\n            if name == \'validation\':\n                return val\n            seed = int(name.removeprefix(\'main_seed\'))\n            return sample(budget, seed)\n\n        with tempfile.TemporaryDirectory() as temp:\n            output = Path(temp)\n            args = run.parse_args([\'--device\', \'cpu\', \'--output\', temp])\n            with mock.patch.object(run.data, \'read_split\', side_effect=read_split), \\\n                 mock.patch.object(run, \'baseline\', return_value=None):\n                run._run_locked(args, cfg, frozen, output)\n                summary = json.loads((output/\'summary.json\').read_text())\n                self.assertEqual(summary[\'state\'], \'completed\')\n                self.assertEqual(summary[\'completed_fits\'], 4)\n                self.assertEqual([r[\'updates\'] for r in summary[\'runs\']], [32, 32, 157, 157])\n                self.assertFalse(summary[\'April_May_evaluated\'])\n                # Rerunning a completed campaign must not invoke training again.\n                with mock.patch.object(run, \'fit_wide\', side_effect=AssertionError(\'Unexpected retraining\')):\n                    run._run_locked(args, cfg, frozen, output)\n                self.assertEqual(json.loads((output/\'summary.json\').read_text())[\'runs\'], summary[\'runs\'])\n\n    def test_baseline_pairing_rejects_different_samples(self):\n        for budget in [200, 1000]:\n            if budget == 200:\n                paths = run.BASE.glob(\'main_recovered/training_*/extracted/main_training/fits/M3_s201_n200/metrics.json\')\n            else:\n                paths = run.BASE.glob(\'continuation_recovered/mixed_n1000/training_*/extracted/continuation/mixed_n1000/training/fits/M3_s201_n1000/metrics.json\')\n            record = json.loads(next(paths).read_text())\n            hashes = {k: record[k] for k in [\'train_ids_sha256\', \'train_input_sha256\',\n                                             \'train_labels_sha256\', \'validation_ids_sha256\']}\n            self.assertIsNotNone(run.baseline(201, budget, hashes))\n            hashes[\'train_ids_sha256\'] = \'different samples\'\n            with self.assertRaises(ValueError):\n                run.baseline(201, budget, hashes)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', '.gitignore': 'runs/\n'}
for name, source in SOURCES.items():
    target = EXP / name
    if target.exists() and target.read_text() != source:
        raise RuntimeError(f'수정된 코드가 있습니다. 덮어쓰지 않습니다: {target}')
    target.write_text(source)
subprocess.run([sys.executable, str(EXP/'smoke_test.py')], check=True)
subprocess.run([sys.executable, str(EXP/'run.py'), '--dry-run'], check=True)


## 2. 공식 데이터 다운로드 및 해시 검증

다운로드가 중간에 끊기면 이 셀을 다시 실행합니다. 완료된 파일은 다시 받지 않습니다.


In [ ]:
SOURCE = ROOT / 'research/2026-09-22_plan_support/data/CESNET-QUICEXT-25-XS.h5'
SOURCE.parent.mkdir(parents=True, exist_ok=True)
SIZE = 3544625537
EXPECTED_SHA = '6b323c4c53b7c9d0ec9011f895b3c5ba6b35247576e41ef19fe84ab8632810ab'
def sha(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()
if not SOURCE.exists() or SOURCE.stat().st_size != SIZE:
    subprocess.run(['curl', '-fL', '--retry', '3', '-C', '-',
        'https://liberouter.org/datazoo/download?bucket=cesnet-quicext-25&file=CESNET-QUICEXT-25-XS.h5',
        '-o', str(SOURCE)], check=True)
assert SOURCE.stat().st_size == SIZE and sha(SOURCE) == EXPECTED_SHA, '데이터 크기/해시 불일치: 학습하지 말고 다운로드를 점검하세요.'
print('공식 데이터 크기·SHA-256 일치')


## 3. 기존과 동일한 데이터 준비

공개 저장소에는 준비 완료 기록만 있고 실제 배열은 없습니다. 첫 실행에 그 기록을 보존한 후 기존 준비 코드로 배열을 만듭니다.
같은 seed에서 `rank < 200`은 `rank < 1000`에 포함됩니다. 재실행 시 완료된 데이터를 확인하고 재사용합니다.
준비 과정에서 4·5월 배열도 생성하지만 **모델 평가에는 사용하지 않습니다**.


In [ ]:
BASE = ROOT / 'experiments/2026-10-30_midterm_restart'
AUDIT = ROOT / 'experiments/2026-10-28_data_audit'
BACKUP = ROOT / 'tmp/m3_width_published'
if not BACKUP.exists():
    assert not (AUDIT/'ppi.npy').exists() and not any((BASE/'prepared').rglob('*.npz')), '이미 실제 데이터가 있습니다. 기존 재현 환경을 임의 초기화하지 마세요.'
    BACKUP.mkdir(parents=True)
# Each move is independently resumable if the notebook stopped during setup.
if not (BACKUP/'audit_manifest.json').exists():
    shutil.move(str(AUDIT/'manifest.json'), str(BACKUP/'audit_manifest.json'))
if not (BACKUP/'prepared').exists():
    shutil.move(str(BASE/'prepared'), str(BACKUP/'prepared'))

subprocess.run([sys.executable, str(AUDIT/'analyze.py'), '--prepare-only'], check=True)
subprocess.run([sys.executable, str(BASE/'data.py'), 'self-test'], check=True)
subprocess.run([sys.executable, str(BASE/'data.py'), 'prepare'], check=True)

old = json.loads((BACKUP/'prepared/manifest.json').read_text())
new = json.loads((BASE/'prepared/manifest.json').read_text())
for key in ['config_sha256', 'source_sha256', 'cache_sha256', 'cohorts']:
    assert old[key] == new[key], f'기존 데이터와 불일치: {key}'
for rel, expected in old['artifacts'].items():
    if rel.endswith('.npz'):
        assert sha(BASE/'prepared'/rel) == expected, f'준비 배열 불일치: {rel}'
print('기존 데이터 캐시·분할·NPZ 해시 모두 일치')


## 4. 200개·1,000개 × seed 2개 = 4회 학습

아래 seed를 `[201, 202, 203, 204, 205]`로 늘리면 총 10회가 됩니다. 기존 완료 결과는 검증 후 재사용합니다.
`MINUTES`는 한 번 실행의 학습 시간 예산이며 GPU 요금 상한이나 자동 해제가 아닙니다.
시간 예산으로 중단되면 같은 셀을 다시 실행하면 됩니다. 런타임 자체가 삭제된 경우에는 먼저 보관한 실행 폴더를 복원해야 합니다.


In [ ]:
SEEDS = [201, 202]
BUDGETS = [200, 1000]
MINUTES = 45
OUTPUT = EXP / 'runs'
command = [sys.executable, str(EXP/'run.py'), '--device', 'cuda',
           '--budgets', *map(str, BUDGETS), '--seeds', *map(str, SEEDS),
           '--minutes', str(MINUTES), '--output', str(OUTPUT)]
result = subprocess.run(command)
SUMMARY = OUTPUT / 'summary.json'
if SUMMARY.exists():
    summary = json.loads(SUMMARY.read_text())
    print('상태:', summary['state'], '완료:', summary['completed_fits'], '/', summary['planned_fits'])
    if summary['state'] == 'budget_stopped':
        print('시간 예산으로 중단됐습니다. 이 셀을 다시 실행해 재개하세요.')
    elif result.returncode:
        raise RuntimeError(summary.get('error') or '학습 실패: 위 로그 확인')
elif result.returncode:
    raise RuntimeError('학습 시작 실패: 위 로그 확인')


## 5. 검증 결과 비교 및 결과 ZIP 다운로드

표의 점수는 모두 **2월 검증**입니다. 기존 보고서의 4월 정답률과 직접 비교하지 마세요.
같은 seed·표본에 대응하는 기존 M3의 검증 성능을 자동으로 가져옵니다.
`기존 환경 일치=False`이면 장비/버전 차이도 섞인 비교이므로 채널만의 효과로 단정하지 않습니다.
50epoch 고정에서는 1,000개 조건의 업데이트 수가 더 많습니다.


In [ ]:
import pandas as pd
summary = json.loads(SUMMARY.read_text())
table = []
for row in summary['runs']:
    old = row['baseline_validation']
    table.append({
        '서비스당 데이터': row['budget'], 'seed': row['seed'],
        '학습 F1(%)': row['train']['macro_f1']*100,
        '확대 검증 F1(%)': row['validation']['macro_f1']*100,
        '확대 검증 정답률(%)': row['validation']['accuracy']*100,
        '기존 검증 F1(%)': None if old is None else old['macro_f1']*100,
        'F1 차이(%p)': row['delta_validation_macro_f1_pp'],
        '선택 epoch': row['selected_epoch'], '업데이트 수': row['updates'],
        '기존 환경 일치': row['baseline_environment_matches'],
    })
display(pd.DataFrame(table))
print('현재 상태:', summary['state'])
print(json.dumps(summary['by_budget'], ensure_ascii=False, indent=2))


In [ ]:
# Includes partial runs and last/best checkpoints so interrupted work can resume.
import zipfile
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
archive = Path('/content') / f'm3_width_results_{stamp}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in EXP.rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts and p.suffix not in ['.lock', '.tmp']:
            z.write(p, p.relative_to(ROOT))
print('결과 ZIP:', archive, 'SHA-256:', sha(archive))
from google.colab import files
files.download(str(archive))
print('다운로드 완료를 확인한 뒤 런타임 연결을 해제하고 삭제하세요.')


## 확인 범위와 한계

- 2026-11-05 Colab T4에서 4회 학습 완료, 저장 체크포인트·예측 재현 검증 통과.
- 학습 코드는 당시 실행본 그대로이며, 4·5월 최종 평가는 별도 `M3_width_final_evaluation.ipynb`로 수행합니다.
- 2026-11-08 보관 모델 4개의 4·5월 평가 총 8건을 완료했습니다. 결과와 환경 차이는 실험 README 및 결과 보고서를 참고하세요.
- 교사/KD, 추가 seed 및 CPU 비용 측정은 수행하지 않았습니다.
- 런타임 종료 전에 `runs` 전체를 보관하세요. `.pt` 하나만으로 재개하지 않습니다.
